<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week4_communication_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 4 — Communicating Without Labels (STARTER)
### The NLP Internship | LinguaAI Labs

This week: pyLDAvis, insight briefs, and the executive presentation.

In [ ]:
import sys, subprocess, importlib
for pkg,imp in [("pyldavis","pyLDAvis"),("scipy","scipy")]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
# pyLDAvis 3.x uses pyLDAvis.sklearn; older used pyLDAvis.lda_model
try:
    import pyLDAvis.sklearn as pyLDAvis_prep
    import pyLDAvis
except ImportError:
    import pyLDAvis.lda_model as pyLDAvis_prep
    import pyLDAvis
print("Setup complete ✅")

## Part 1 — Rebuild the LDA Model from Week 3

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/support_tickets.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/nlp-internship-in-a-book/main/data/support_tickets.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload support_tickets.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        tickets = [
            'My account login is not working, please help',
            'I was charged twice for my subscription this month',
            'The app crashes whenever I try to upload a file',
            'How do I cancel my subscription?',
            'Great service, very happy with the product!',
            'I cannot reset my password, the email never arrives',
            'The dashboard is loading very slowly today',
            'I need a refund for my last order',
            'Feature request: can you add dark mode?',
            'Error 500 when trying to export my data',
        ] * 50
        np.random.shuffle(tickets)
        df = pd.DataFrame({
            'ticket_id':  range(1, 501),
            'text':       tickets[:500],
            'category':   np.random.choice(['billing','technical','account','general'], 500),
            'sentiment':  np.random.choice(['positive','negative','neutral'], 500, p=[0.3,0.4,0.3]),
            'priority':   np.random.choice(['low','medium','high'], 500),
        })
        print('Sample dataset ready (500 tickets) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


## Part 2 — pyLDAvis Interactive Visualisation

> ⏸️ **Pause and Predict**
> Before generating the visualisation: which two topics do you expect to overlap most (sit closest in the 2D map)?

In [ ]:
# Create pyLDAvis display
print("Preparing pyLDAvis (may take 1–2 minutes)...")
lda_display = pyLDAvis_prep.prepare(lda, dtm, lda_vec, mds="tsne")
pyLDAvis.save_html(lda_display, "outputs/topic_visualisation.html")
print("Saved: outputs/topic_visualisation.html")
print("Open this file in your browser and explore the topic map.")

## Part 3 — Inter-Topic Distances

In [ ]:
from scipy.spatial.distance import jensenshannon

topic_word_probs = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
n_topics = 8
print("Inter-topic Jensen-Shannon distances (lower = more similar):\n")
print(f"{'':>10}", end="")
for j in range(n_topics):
    print(f"  T{j+1}", end="")
print()
for i in range(n_topics):
    print(f"  Topic {i+1}:", end="")
    for j in range(n_topics):
        if i == j:
            print("  ---", end="")
        else:
            d = jensenshannon(topic_word_probs[i], topic_word_probs[j])
            print(f"  {d:.2f}", end="")
    print()

## Part 4 — Channel Analysis Charts

In [ ]:
TOPIC_NAMES = {
    0:"Account\nAccess", 1:"Billing &\nPayments", 2:"Technical\nIssues",
    3:"Delivery &\nLogistics", 4:"Feedback", 5:"Escalation\n& Urgency",
    6:"Refund\nRequests", 7:"Product\nIssues",
}
doc_topic_probs = lda.transform(dtm)
df["dominant_topic"] = doc_topic_probs.argmax(axis=1)

topic_by_channel = df.groupby(["channel","dominant_topic"]).size().unstack(fill_value=0)
topic_by_channel_pct = topic_by_channel.div(topic_by_channel.sum(axis=1), axis=0) * 100

# YOUR CODE HERE — create a grouped bar chart of topic distribution by channel
# Title should lead with the finding, e.g.:
# "Social Channel Has 3× More Escalation Tickets Than Email"


## Part 5 — Insight Brief

> Write a Pyramid Principle brief in the markdown cell below.
> Lead with the finding — not the methodology.

---
## CustomerCare-50K: Topic Analysis Brief

**Top Finding:** [YOUR MOST IMPORTANT FINDING IN ONE SENTENCE]

**High Confidence:**
- [Finding 1 with supporting evidence]
- [Finding 2 with supporting evidence]

**Medium Confidence:**
- [Finding with caveat]

**What We Do NOT Know:**
- [One honest gap]

**Recommended Next Actions:**
1. [Action 1]
2. [Action 2]
3. [Action 3]

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Showing word clouds as the main finding | Word clouds are decoration, not analysis. The most common word is not the most important finding. | Every visualisation must answer a specific question. State the question in the title. |
| Presenting all topics with equal weight | Not all topics matter equally. Some affect 40% of tickets; others affect 2%. Weight your findings by business impact. | Lead with the highest-impact finding, not the most interesting one |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Build three visualisations that answer three different business questions. Each chart title must state the finding, not the variable.

> 🔬 **Advanced Path**
> Write a one-page executive memo summarising the topic analysis. No charts — plain English. What should the product team do differently based on this analysis?

## ✅ What You Can Do After This Week

- Visualise an LDA model with pyLDAvis
- Write a Pyramid Principle insight brief
- Build channel-specific comparison charts

---
## ✅ Week 4 Complete
**Next:** `week5_first_classifier_STARTER.ipynb`

---
*Add `week4_communication_STARTER.ipynb` to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 4, you can now:

- Build visualisations that answer specific business questions
- Communicate topic model results to a non-technical audience without losing accuracy
- Write a data story with situation, finding, implication, and recommendation
- Adapt your communication style for technical vs non-technical audiences

📤 **GitHub:** Push week4_communication_STARTER.ipynb and slide images. Commit: "Week 4: Topic analysis communicated"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
